# Ukrainian to English Translation Pipeline (GPU)

This notebook translates all Ukrainian design education texts to English using GPU acceleration.

## VS Code Integration
To connect VS Code to this Colab runtime:
1. Install the 'Remote - SSH' extension in VS Code
2. Run the cell below to get SSH connection details
3. Or use the Jupyter extension with Colab kernel

**Expected time: ~30-60 minutes for 4,027 files on T4 GPU**

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q transformers sentencepiece sacremoses torch accelerate
!pip install -q tqdm

In [ ]:
# Mount Google Drive (for data storage)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration
import os

# Set paths - UPDATE THESE to match your Drive structure
DRIVE_BASE = '/content/drive/MyDrive/thesis_data'
INPUT_DIR = f'{DRIVE_BASE}/text_by_level'  # Ukrainian texts
OUTPUT_DIR = f'{DRIVE_BASE}/text_english'  # English translations

# Or use local upload
USE_LOCAL_UPLOAD = False  # Set True to upload ZIP file instead

# Create output directory
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Input: {INPUT_DIR}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
# Alternative: Upload ZIP file directly
if USE_LOCAL_UPLOAD:
    from google.colab import files
    print('Upload text_by_level.zip file:')
    uploaded = files.upload()
    
    # Extract
    !unzip -q text_by_level.zip -d /content/
    INPUT_DIR = '/content/text_by_level'
    OUTPUT_DIR = '/content/text_english'
    os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
import torch
from transformers import MarianMTModel, MarianTokenizer
from pathlib import Path
import re
from tqdm.auto import tqdm
import gc

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# Load translation model
MODEL_NAME = 'Helsinki-NLP/opus-mt-uk-en'
print(f'Loading model: {MODEL_NAME}')

tokenizer = MarianTokenizer.from_pretrained(MODEL_NAME)
model = MarianMTModel.from_pretrained(MODEL_NAME).to(device)
model.eval()

print('Model loaded successfully!')

In [ ]:
def chunk_text(text, max_length=400):
    """Split text into chunks for translation."""
    if not text or len(text) <= max_length:
        return [text] if text else []
    
    chunks = []
    sentences = re.split(r'(?<=[.!?])\s+', text)
    current_chunk = ''
    
    for sentence in sentences:
        if len(sentence) > max_length:
            # Split long sentence by words
            words = sentence.split()
            for word in words:
                if len(current_chunk) + len(word) + 1 <= max_length:
                    current_chunk += ' ' + word if current_chunk else word
                else:
                    if current_chunk:
                        chunks.append(current_chunk.strip())
                    current_chunk = word
        elif len(current_chunk) + len(sentence) + 1 <= max_length:
            current_chunk += ' ' + sentence if current_chunk else sentence
        else:
            if current_chunk:
                chunks.append(current_chunk.strip())
            current_chunk = sentence
    
    if current_chunk:
        chunks.append(current_chunk.strip())
    
    return chunks


def translate_batch(texts, batch_size=16):
    """Translate a batch of texts using GPU."""
    if not texts:
        return []
    
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        # Filter empty texts
        batch = [t for t in batch if t and t.strip()]
        if not batch:
            continue
        
        # Tokenize
        inputs = tokenizer(batch, return_tensors='pt', padding=True, 
                          truncation=True, max_length=512).to(device)
        
        # Translate
        with torch.no_grad():
            translated = model.generate(**inputs, max_length=512)
        
        # Decode
        decoded = tokenizer.batch_decode(translated, skip_special_tokens=True)
        results.extend(decoded)
    
    return results


def translate_text(text):
    """Translate a single text document."""
    if not text or not text.strip():
        return ''
    
    # Split into chunks
    chunks = chunk_text(text, max_length=350)
    
    if not chunks:
        return ''
    
    # Translate chunks
    translated_chunks = translate_batch(chunks, batch_size=8)
    
    return ' '.join(translated_chunks)


print('Translation functions ready!')

In [ ]:
# Test translation
test_texts = [
    "Метою курсу є формування знань студентів про основи дизайну.",
    "Історія українського мистецтва та культури.",
    "Комп'ютерні технології в графічному дизайні.",
]

print('Testing translation...')
for text in test_texts:
    translated = translate_text(text)
    print(f'UK: {text}')
    print(f'EN: {translated}')
    print()

In [ ]:
# Find all text files
input_path = Path(INPUT_DIR)
text_files = list(input_path.rglob('*.txt'))
print(f'Found {len(text_files)} text files to translate')

In [ ]:
# Main translation loop
output_path = Path(OUTPUT_DIR)

success = 0
errors = 0
total_chars = 0

for txt_file in tqdm(text_files, desc='Translating'):
    try:
        # Read Ukrainian text
        with open(txt_file, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read()
        
        total_chars += len(text)
        
        # Translate
        if text.strip():
            translated = translate_text(text)
        else:
            translated = ''
        
        # Save to output (preserve directory structure)
        rel_path = txt_file.relative_to(input_path)
        out_file = output_path / rel_path
        out_file.parent.mkdir(parents=True, exist_ok=True)
        
        with open(out_file, 'w', encoding='utf-8') as f:
            f.write(translated)
        
        success += 1
        
        # Clear GPU memory periodically
        if success % 100 == 0:
            torch.cuda.empty_cache()
            gc.collect()
    
    except Exception as e:
        errors += 1
        print(f'Error: {txt_file}: {e}')

print(f'\n\nTranslation complete!')
print(f'Success: {success}')
print(f'Errors: {errors}')
print(f'Total characters processed: {total_chars:,}')

In [ ]:
# Verify output
output_files = list(Path(OUTPUT_DIR).rglob('*.txt'))
print(f'Output files created: {len(output_files)}')

# Show sample translation
if output_files:
    sample = output_files[0]
    print(f'\nSample output ({sample.name}):')
    with open(sample, 'r') as f:
        print(f.read()[:500])

In [ ]:
# Download results as ZIP (if using local upload)
if USE_LOCAL_UPLOAD:
    !zip -r text_english.zip {OUTPUT_DIR}
    from google.colab import files
    files.download('text_english.zip')
else:
    print(f'Results saved to Google Drive: {OUTPUT_DIR}')

## VS Code Remote Connection (Optional)

To connect VS Code directly to this Colab runtime, run the cell below.
This uses `colab-ssh` for SSH tunneling.

In [ ]:
# Optional: Set up VS Code SSH connection
# Uncomment to use

# !pip install colab-ssh --quiet
# from colab_ssh import launch_ssh_cloudflared
# launch_ssh_cloudflared(password="your_password_here")

# Then in VS Code:
# 1. Install 'Remote - SSH' extension
# 2. Use the cloudflared URL provided above
# 3. Connect and open /content folder

## Summary Statistics

In [ ]:
# Final summary
import os

def get_dir_size(path):
    total = 0
    for dirpath, _, filenames in os.walk(path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            total += os.path.getsize(fp)
    return total

input_size = get_dir_size(INPUT_DIR)
output_size = get_dir_size(OUTPUT_DIR)

print('=' * 50)
print('TRANSLATION SUMMARY')
print('=' * 50)
print(f'Input files: {len(text_files)}')
print(f'Output files: {len(output_files)}')
print(f'Input size: {input_size / 1024 / 1024:.1f} MB')
print(f'Output size: {output_size / 1024 / 1024:.1f} MB')
print(f'Characters translated: {total_chars:,}')
print('=' * 50)